In [11]:
#pip install torch torchvision torch-geometric
#pip install rdkit-pypi deepchem chemprop==1.6.1

In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GCNConv, global_mean_pool
from rdkit import Chem, RDLogger
from sklearn.model_selection import train_test_split
# import deepchem as dc
from rdkit.Chem import rdchem
from rdkit.Chem import AllChem
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score, roc_auc_score
from rdkit import Chem
from rdkit import DataStructs


RDLogger.DisableLog('rdApp.warning')  # 禁用所有RDKit警告
# 检查 GPU 是否可用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cpu


In [13]:
# 配置参数
BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 0.001
HIDDEN_DIM = 256
DROPOUT = 0.2
TARGET_COLS = ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 
              'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE',
              'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


In [14]:
# 1. 数据预处理（每个任务单独处理）

def is_valid_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol is not None and mol.GetNumAtoms() > 1
    except Exception:
        return False

def load_single_task_data(path,target_col):

    df = pd.read_csv(path)
    
    # 过滤当前任务的缺失值
    df = df.dropna(subset=[target_col])
    print(f"\nTask {target_col} Valid sample size: {len(df)}")
       
 # 过滤有效的 SMILES 并保留索引
    valid_smiles_indices = df['smiles'].apply(is_valid_smiles)
    
    # 过滤目标列值为 0 或 1 的样本
    valid_target_indices = df[target_col].isin([0, 1])
    
    # 合并两个条件，得到最终有效索引
    valid_indices = valid_smiles_indices & valid_target_indices
    valid_df = df[valid_indices]
    
    # 提取有效的 SMILES 和目标值
    valid_smiles = valid_df['smiles'].values
    y = valid_df[target_col].values.astype(np.float32)
    
    return valid_smiles, y

In [15]:


def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # 手性编码映射字典
    cip_code_mapping = {'R': 0, 'S': 1, 'r': 2, 's': 3, '': -1}
    
    # 原子特征提取
    atom_features = []
    for atom in mol.GetAtoms():
        # 处理手性编码（数值化）
        cip_code = atom.GetProp("_CIPCode") if atom.HasProp("_CIPCode") else ""
        cip_value = cip_code_mapping.get(cip_code.upper(), -1)  # 转换为数值
        
        features = [
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetTotalDegree(),
            atom.GetImplicitValence(),
            atom.GetExplicitValence(),
            int(atom.GetIsAromatic()),
            atom.GetHybridization().real,
            atom.GetTotalNumHs(),
            int(atom.IsInRing()),
            atom.GetMass() / 100.0,
            atom.GetNumRadicalElectrons(),
            atom.GetChiralTag().real,
            cip_value,  # 使用数值编码后的手性特征
            atom.GetFormalCharge(),
            atom.GetNumExplicitHs(),
            atom.GetIsotope(),
            int(atom.HasOwningMol()),
            *[int(atom.IsInRingSize(i)) for i in range(3, 8)],
        ]
        atom_features.append(features)
        
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
        arr = np.zeros((1024,), dtype=int)
        DataStructs.ConvertToNumpyArray(fp, arr)
        
        atom_features[-1].extend(arr)  # 将指纹特征添加到原子特征中

    
    # 边特征提取
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.extend([[i, j], [j, i]])
        
        bond_features = [
            bond.GetBondTypeAsDouble(),
            int(bond.IsInRing()),
            int(bond.GetIsConjugated()),
            bond.GetBondDir().real,
            bond.GetStereo().real,
            bond.GetBondTypeAsDouble() * int(bond.IsInRing()),
            int(bond.GetIsAromatic()),
        ]
        edge_attr.extend([bond_features, bond_features])
    


    return Data(
        x=torch.tensor(atom_features, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float)
    )


In [16]:
# from rdkit import Chem
# from rdkit.Chem import Descriptors, rdMolDescriptors, rdchem
# from rdkit.Chem.rdchem import HybridizationType, BondType, BondDir, ChiralType
# import torch
# from torch_geometric.data import Data
# import numpy as np

# def get_atom_features(atom):
#     """提取原子级特征（总计48维）"""
#     features = []
    
#     # 基础属性
#     features += [
#         atom.GetAtomicNum(),                        # 原子序数
#         atom.GetDegree(),                           # 直接连接原子数
#         atom.GetTotalDegree(),                      # 总连接数
#         atom.GetImplicitValence(),                  # 隐式价
#         atom.GetExplicitValence(),                  # 显式价
#         int(atom.GetIsAromatic()),                 # 芳香性
#         atom.GetTotalNumHs(),                       # 总H数
#     ]
    
#     # 杂化与手性
#     features += [
#         atom.GetHybridization().real,               # 杂化类型编码
#         int(atom.GetHybridization() == HybridizationType.SP3),
#         int(atom.GetHybridization() == HybridizationType.SP2),
#         atom.GetChiralTag().real,                   # 手性编码
#     ]
    
#     # 电子特性
#     features += [
#         atom.GetFormalCharge(),                     # 形式电荷
#         atom.GetNumRadicalElectrons(),               # 自由基电子
#         atom.GetIsotope(),                          # 同位素质量
#         rdchem.GetPeriodTable().GetNOuterElecs(atom.GetAtomicNum()), # 外层电子
#         rdchem.GetPeriodTable().GetElectronegativity(atom.GetAtomicNum()), # 电负性
#     ]
    
#     # 物理化学性质
#     features += [
#         rdchem.GetPeriodTable().GetRvdw(atom.GetAtomicNum())/10,  # 范德华半径
#         rdchem.GetPeriodTable().GetRcovalent(atom.GetAtomicNum())/10, # 共价半径
#         atom.GetMass()/100.0,                       # 原子质量
#     ]
    
#     # 环信息
#     features += [
#         int(atom.IsInRing()),                       # 是否在环中
#         *[int(atom.IsInRingSize(i)) for i in range(3, 9)],  # 3-8元环检测
#     ]
#     # 药效团特征
#     features += [
#         int(atom.GetAtomicNum() in [7,8,15,16]),     # 常见氢键受体
#         int(atom.GetTotalNumHs() > 0),               # 氢键供体
#     ]
    
#     # 立体化学
#     cip_code = atom.GetProp("_CIPCode") if atom.HasProp("_CIPCode") else ""
#     features += [
#         int(cip_code in ['R','S']),                  # 是否手性中心
#         int(atom.HasProp("_ChiralityPossible")),     # 潜在手性
#     ]
    
#     return features

# def get_bond_features(bond, mol):
#     """提取键级特征（总计25维）"""
#     features = []
    
#     # 基础属性
#     features += [
#         bond.GetBondType().real,                    # 键类型
#         int(bond.IsInRing()),                       # 是否在环中
#         int(bond.GetIsConjugated()),                # 是否共轭
#         bond.GetBondDir().real,                     # 键方向
#         bond.GetStereo().real,                      # 立体化学
#     ]
    
#     # 物理化学
#     bond_order = bond.GetBondTypeAsDouble()
#     features += [
#         bond_order,                                 # 键级
#         bond.GetLength() if mol.GetConformer() else 0.0, # 键长(需3D结构)
#         bond.GetValenceContrib(0),                  # 原子1的价贡献
#         bond.GetValenceContrib(1),                  # 原子2的价贡献
#     ]
    
#     # 电子特性
#     features += [
#         int(bond.GetIsAromatic()),                 # 芳香键
#         bond.GetBondType().GetNumRadicalElectrons(),# 自由基电子
#     ]
    
#     # 拓扑特性
#     atom1 = bond.GetBeginAtom()
#     atom2 = bond.GetEndAtom()
#     features += [
#         atom1.GetAtomicNum() + atom2.GetAtomicNum(),# 原子序数和
#         abs(atom1.GetAtomicNum() - atom2.GetAtomicNum()), # 原子序数差
#     ]
    
#     # 3D几何特征（需要3D结构）
#     if mol.GetConformer():
#         coords = mol.GetConformer().GetPositions()
#         v1 = coords[bond.GetBeginAtomIdx()] - coords[bond.GetEndAtomIdx()]
#         features += [
#             np.linalg.norm(v1),                     # 键长(冗余特征)
#             v1[0], v1[1], v1[2],                    # 向量分量
#         ]
#     else:
#         features += [0.0]*4
    
#     return features

# def get_molecular_features(mol):
#     """提取分子级特征（总计127维）"""
#     features = []
    
#     # 基础描述符
#     features += [
#         Chem.Descriptors.MolWt(mol)/1000,           # 分子量
#         Chem.Descriptors.TPSA(mol)/100,             # 极性表面积
#         Chem.Descriptors.MolLogP(mol),              # LogP
#         Chem.Descriptors.NumHDonors(mol),           # 氢键供体
#         Chem.Descriptors.NumHAcceptors(mol),        # 氢键受体
#     ]
    
#     # 拓扑描述符
#     features += [
#         rdMolDescriptors.CalcNumRotatableBonds(mol),# 可旋转键
#         rdMolDescriptors.CalcNumHeterocycles(mol),  # 杂环数
#         rdMolDescriptors.CalcNumAromaticRings(mol), # 芳香环
#         rdMolDescriptors.CalcNumSaturatedRings(mol),# 饱和环
#         rdMolDescriptors.CalcNumHeteroatoms(mol),   # 杂原子数
#     ]
    
#     # 电荷相关
#     features += [
#         Chem.Descriptors.NumRadicalElectrons(mol),  # 自由基电子
#         Chem.Descriptors.NHOHCount(mol),            # NH或OH基团
#         Chem.Descriptors.NOCount(mol),              # 硝基/氧基数
#     ]
    
#     # 3D描述符（需要3D结构）
#     if mol.GetConformer():
#         moments = rdMolDescriptors.CalcPMI1(mol), rdMolDescriptors.CalcPMI2(mol)
#         features += [
#             rdMolDescriptors.CalcInertialShapeFactor(mol),
#             moments[0],
#             moments[1],
#             rdMolDescriptors.CalcAsphericity(mol),
#         ]
#     else:
#         features += [0.0]*4
    
#     # Mordred描述符（需安装mordred）
#     try:
#         calc = Calculator(descriptors)
#         mordred_features = calc(mol).fill_missing()
#         features += list(mordred_features.values)[:50]  # 取前50个
#     except:
#         features += [0.0]*50
    
#     return features

# def smiles_to_graph(smiles):
#     """生成包含全量特征的图结构"""
#     mol = Chem.MolFromSmiles(smiles)
#     if not mol:
#         return None
    
#     # 生成3D结构
#     mol = Chem.AddHs(mol)
#     Chem.AllChem.EmbedMolecule(mol)
    
#     # 原子特征
#     atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
    
#     # 键特征
#     edge_index = []
#     edge_attr = []
#     for bond in mol.GetBonds():
#         i = bond.GetBeginAtomIdx()
#         j = bond.GetEndAtomIdx()
#         features = get_bond_features(bond, mol)
#         edge_index.extend([[i, j], [j, i]])
#         edge_attr.extend([features, features])
    
#     # 分子级特征
#     mol_features = get_molecular_features(mol)
    
#     return Data(
#         x=torch.tensor(atom_features, dtype=torch.float),
#         edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
#         edge_attr=torch.tensor(edge_attr, dtype=torch.float),
#         y=torch.tensor([mol_features], dtype=torch.float)  # 假设任务需要分子特征
#     )

# # 使用示例
# if __name__ == "__main__":
#     graph = smiles_to_graph("CCO")
#     print(f"原子特征维度: {graph.x.shape[1]}")    # 应输出48
#     print(f"边特征维度: {graph.edge_attr.shape[1]}")  # 应输出25
#     print(f"分子特征维度: {graph.y.shape[1]}")     # 应输出127

In [17]:
# 3. 自定义数据集类
class SingleTaskDataset(Dataset):
    def __init__(self, smiles_list, y):
        super().__init__()
        self.smiles_list = smiles_list
        self.y = torch.tensor(y, dtype=torch.float)
        
    def len(self):
        return len(self.smiles_list)
    
    def get(self, idx):
        data = smiles_to_graph(self.smiles_list[idx])
        data.y = self.y[idx].unsqueeze(0)  # 保持二维形状
        return data


In [18]:
class SingleTaskGCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, edge_dim):
        super().__init__()
        # 第一层：融合边特征到节点特征
        self.edge_processor = torch.nn.Linear(edge_dim, input_dim)
        
        # 保持原有结构
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, 1)

        # 初始化参数
        self._init_weights()

    def _init_weights(self):
        for module in [self.edge_processor, self.conv1, self.conv2, self.lin]:
            if hasattr(module, 'weight'):
                torch.nn.init.xavier_uniform_(module.weight)
            if hasattr(module, 'bias') and module.bias is not None:
                module.bias.data.fill_(0.01)

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        
        # === 边特征融合（不改变输入结构）===
        # 将边特征映射到节点维度
        edge_emb = self.edge_processor(edge_attr)
        
        # 通过聚合边特征增强节点特征
        row, col = edge_index
        aggregated_edge = torch.zeros_like(x).index_add_(0, col, edge_emb)
        x = x + aggregated_edge  # 保持x维度不变

        # === 保持原有GCN结构 ===
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=DROPOUT, training=self.training)
        
        # 第二层使用边权重
        edge_weight = edge_attr.mean(dim=1)  # 将多维边特征转为标量权重
        x = self.conv2(x, edge_index, edge_weight=edge_weight)
        
        # === 保持原有池化结构 ===
        if hasattr(data, 'batch') and data.batch is not None:
            x = global_mean_pool(x, data.batch)
        else:
            x = x.mean(dim=0, keepdim=True)
            
        x = self.lin(x)
        return x.squeeze(-1)

In [19]:
def train_single_task(target_col):
    # 加载数据
    smiles, y = load_single_task_data("tox21_cleaned.csv", target_col)

    # 划分数据集
    X_train, X_test, y_train, y_test = train_test_split(
        smiles, y, test_size=0.2, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_test, y_test, test_size=0.5, random_state=42
    )

    # 创建数据集
    train_dataset = SingleTaskDataset(X_train.tolist(), y_train)
    val_dataset = SingleTaskDataset(X_val.tolist(), y_val)
    test_dataset = SingleTaskDataset(X_test.tolist(), y_test)

    # 创建数据加载器
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    # 初始化模型并移动到 GPU
    model = SingleTaskGCN(
        input_dim=train_dataset[0].x.shape[1],
        hidden_dim=HIDDEN_DIM,
        edge_dim=7  # 边特征维度
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = torch.nn.BCEWithLogitsLoss()

    best_auc = 0
    patience = 5  # 早停的容忍次数
    patience_counter = 0

    for epoch in range(EPOCHS):
        # 训练阶段
        model.train()
        total_loss = 0
        for data in train_loader:
            # 将数据移动到 GPU
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # 验证阶段
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)  # 将数据移动到 GPU
                pred = model(data)
                val_preds.append(pred.sigmoid().cpu().numpy())  # 移动到 CPU
                val_labels.append(data.y.cpu().numpy())

        val_auc = roc_auc_score(
            np.concatenate(val_labels), np.concatenate(val_preds)
        )
        print(f"Task {target_col} | Epoch {epoch+1}/{EPOCHS} | "
              f"Train Loss: {total_loss/len(train_loader):.4f} | "
              f"Val AUC: {val_auc:.4f}")

        # 检查是否需要早停
        if val_auc > best_auc:
            best_auc = val_auc
            patience_counter = 0
            torch.save(model.state_dict(), f"best_model_{target_col}.pth")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break

    # 测试阶段
    model.load_state_dict(torch.load(f"best_model_{target_col}.pth"))
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)  # 将数据移动到 GPU
            pred = model(data)
            test_preds.append(pred.sigmoid().cpu().numpy())  # 移动到 CPU
            test_labels.append(data.y.cpu().numpy())

    test_preds = np.concatenate(test_preds)
    test_labels = np.concatenate(test_labels)

    # 计算评估指标
    recall = recall_score(test_labels, test_preds.round())
    precision = precision_score(test_labels, test_preds.round())
    f1 = f1_score(test_labels, test_preds.round())
    accuracy = accuracy_score(test_labels, test_preds.round())
    auc = roc_auc_score(test_labels, test_preds)
    
    print(f'TEST AUC: {auc}')

    return {
        "Recall": recall,
        "Precision": precision,
        "F1 Score": f1,
        "Accuracy": accuracy,
        "AUC": auc
    }

In [20]:
if __name__ == "__main__":
    results = {}
    for target_col in TARGET_COLS:
        metrics = train_single_task(target_col)
        results[target_col] = metrics


Task NR-AR Valid sample size: 7432
Task NR-AR | Epoch 1/30 | Train Loss: 0.1627 | Val AUC: 0.8130
Task NR-AR | Epoch 2/30 | Train Loss: 0.1100 | Val AUC: 0.8537
Task NR-AR | Epoch 3/30 | Train Loss: 0.0990 | Val AUC: 0.8478
Task NR-AR | Epoch 4/30 | Train Loss: 0.0926 | Val AUC: 0.8337
Task NR-AR | Epoch 5/30 | Train Loss: 0.0780 | Val AUC: 0.8613
Task NR-AR | Epoch 6/30 | Train Loss: 0.0775 | Val AUC: 0.8574
Task NR-AR | Epoch 7/30 | Train Loss: 0.0603 | Val AUC: 0.8259
Task NR-AR | Epoch 8/30 | Train Loss: 0.0569 | Val AUC: 0.8535
Task NR-AR | Epoch 9/30 | Train Loss: 0.0655 | Val AUC: 0.8551
Task NR-AR | Epoch 10/30 | Train Loss: 0.0436 | Val AUC: 0.8207
Early stopping triggered after 10 epochs.
TEST AUC: 0.7910157830837717

Task NR-AR-LBD Valid sample size: 6895
Task NR-AR-LBD | Epoch 1/30 | Train Loss: 0.1461 | Val AUC: 0.6526
Task NR-AR-LBD | Epoch 2/30 | Train Loss: 0.0935 | Val AUC: 0.7803
Task NR-AR-LBD | Epoch 3/30 | Train Loss: 0.0893 | Val AUC: 0.7706
Task NR-AR-LBD | Epoc

d:\Anaconda\envs\chem311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


TEST AUC: 0.8076615831517793

Task SR-MMP Valid sample size: 5914
Task SR-MMP | Epoch 1/30 | Train Loss: 0.4454 | Val AUC: 0.7766
Task SR-MMP | Epoch 2/30 | Train Loss: 0.3456 | Val AUC: 0.8441
Task SR-MMP | Epoch 3/30 | Train Loss: 0.3177 | Val AUC: 0.8729
Task SR-MMP | Epoch 4/30 | Train Loss: 0.2549 | Val AUC: 0.8877
Task SR-MMP | Epoch 5/30 | Train Loss: 0.2279 | Val AUC: 0.8989
Task SR-MMP | Epoch 6/30 | Train Loss: 0.1933 | Val AUC: 0.8974
Task SR-MMP | Epoch 7/30 | Train Loss: 0.1756 | Val AUC: 0.8999
Task SR-MMP | Epoch 8/30 | Train Loss: 0.1343 | Val AUC: 0.8891
Task SR-MMP | Epoch 9/30 | Train Loss: 0.1132 | Val AUC: 0.8897
Task SR-MMP | Epoch 10/30 | Train Loss: 0.0790 | Val AUC: 0.8928
Task SR-MMP | Epoch 11/30 | Train Loss: 0.0700 | Val AUC: 0.8872
Task SR-MMP | Epoch 12/30 | Train Loss: 0.0541 | Val AUC: 0.8856
Early stopping triggered after 12 epochs.
TEST AUC: 0.9127877237851663

Task SR-p53 Valid sample size: 6902
Task SR-p53 | Epoch 1/30 | Train Loss: 0.2632 | Val AUC

In [ ]:
# 转换为 pandas DataFrame
results_df = pd.DataFrame(results).T

# 计算 mean 和 median
mean_auc = results_df['AUC'].mean()
median_auc = results_df['AUC'].median()

# 打印结果
print(f"Mean AUC: {mean_auc:.4f}")
print(f"Median AUC: {median_auc:.4f}")

display(results_df.round(4))

Mean AUC: 0.8234
Median AUC: 0.8249


,Recall,Precision,F1 Score,Accuracy,AUC
NR-AR,0.4000,1.0000,0.5714,0.9717,0.7910
NR-AR-LBD,0.4762,0.7692,0.5882,0.9797,0.8993
NR-AhR,0.4783,0.5789,0.5238,0.9100,0.8888
NR-Aromatase,0.1379,0.5000,0.2162,0.9510,0.8578
NR-ER,0.2099,0.7083,0.3238,0.8873,0.7336
NR-ER-LBD,0.3611,0.8667,0.5098,0.9647,0.8682
NR-PPAR-gamma,0.2500,0.3333,0.2857,0.9619,0.7054
SR-ARE,0.4175,0.5375,0.4699,0.8361,0.7856
SR-ATAD5,0.2069,0.5455,0.3000,0.9612,0.8421
SR-HSE,0.0000,0.0000,0.0000,0.9315,0.8077


In [22]:
# print("\nFinal outputs summary:")
# for target, auc in results.items():
#     print(f"{target}: {auc:.4f}")
# print(f"Mean AUC: {np.mean(list(results.values())):.4f}")